# The Ultimate AI Engineering Interview QA Guide

This notebook contains the questions formulated in the main README and provides rigorous, expert-level answers for each of them. You can use the empty code cells below certain topics to practice implementations!


## 1. Core Must-Know Concepts


### Q: LLM Context: Standard pre-training vs. Instruction pre-training algorithms.
**Answer:** 
- **Standard Pre-Training:** The model is trained on a massive corpus of raw internet text using an auto-regressive objective (predicting the next token). It learns grammar, facts, and reasoning but acts purely as a text-completer.
- **Instruction Tuning (SFT):** The pre-trained model is further trained on high-quality pairs of `(Instruction, Response)`. This teaches the model to behave like a helpful assistant that answers queries rather than just predicting the next likely word on a webpage.


### Q: RAG Architecture: The baseline components (Embed -> Retrieve -> Generate).
**Answer:**
1. **Embed:** User documents are chunked and passed through an embedding model (e.g., text-embedding-3-small) to create high-dimensional vectors, which are stored in a Vector Database.
2. **Retrieve:** When a query is asked, the query is embedded, and a cosine-similarity search retrieves the Top-K most relevant document chunks.
3. **Generate:** The retrieved chunks are injected into the prompt context of an LLM, which then generates a formulated answer based *only* on that context.


### Q: Tool Use & Function Calling: Enabling deterministic actions via LLMs.
**Answer:** Tool use allows an LLM to interact with the outside world. Instead of answering directly, the model emits a structured JSON object representing a function signature (e.g., `{"name": "get_weather", "args": {"location": "NYC"}}`). The application parses this, runs the deterministic local code, and returns the output back to the LLM to summarize.


### Q: MCP (Model Context Protocol).
**Answer:** MCP is an emerging open standard introduced by Anthropic that standardizes how foundation models communicate with external data sources and tools. Instead of custom-building tool integrations for every LLM, MCP provides a unified API for servers to expose data and tools securely.


### Q: Agentic Workflows: Loops, reflection, and multi-agent coordination.
**Answer:** Unlike a simple linear prompt, an agentic workflow involves a loop where the LLM reasons, decides on an action, observes the result, and loops back (e.g., ReAct). Reflection involves prompting the model to grade its own previous outputs. Multi-agent coordination assigns specific personas (e.g., Planner, Coder, Reviewer) to different LLM calls that communicate to solve a massive task.


### Q: PEFT / LoRA: Parameter-Efficient Fine-Tuning basics.
**Answer:** Instead of updating billions of parameters during fine-tuning (which requires massive VRAM), LoRA freezes the base model weights and injects trainable rank-decomposition matrices into the layers. This drastically reduces the number of trainable parameters (often by 10,000x) while retaining >95% of full fine-tuning performance.


### Q: Quantization: Moving from FP16 to INT8/INT4.
**Answer:** Quantization compresses the mathematical precision of the model weights. By casting 16-bit floats to 8-bit or 4-bit integers, you cut the VRAM requirements by 2x or 4x. While it introduces slight noise (truncation errors), techniques like AWQ or GPTQ maintain high accuracy, enabling large models to run on consumer GPUs.


In [ ]:
# Practice Area: Write a simple script here to mock an LLM function call JSON response and parse it.


## 2. LLM Fundamentals & Architecture


### Q: What are foundation models, and how do they differ from task-specific ML models?
**Answer:** Foundation models are extremely large neural networks trained on vast amounts of unlabelled data, capable of adapting to a wide range of downstream tasks (zero-shot or few-shot). Task-specific ML models (like an XGBoost classifier for churn) are trained for one single, narrow objective and cannot generalize.


### Q: Explain Tokenization and how it inherently impacts math and coding.
**Answer:** Tokenization splits text into subword chunks (e.g., BPE). If a tokenizer splits numbers randomly (e.g., 3892 as `[38] [92]`), it makes mathematical reasoning difficult because the spatial structure is destroyed. This is why newer tokenizers (like Llama 3's) allocate specific tokens for individual digits and vast coding vocabularies.


### Q: Explain Causal Masking.
**Answer:** In the Transformer decoder, causal masking ensures that when predicting the token at position `i`, the attention mechanism can only look at tokens up to `i-1`. It sets the attention weights for future tokens to negative infinity, preventing the model from "cheating" by looking ahead.


### Q: What is the KV Cache and PagedAttention?
**Answer:** The KV Cache stores the Key and Value matrices of previously generated tokens so they don't have to be recomputed at every step. *PagedAttention* (used in vLLM) solves memory fragmentation by partitioning the KV cache into fixed-size blocks (like OS virtual memory), drastically improving memory usage and allowing batch sizes to scale 3x-4x.


### Q: Logits, Temperature, and standard samplers.
**Answer:** 
- **Logits:** The raw, unnormalized scores outputted by the final layer for every token in the vocabulary.
- **Temperature:** Divides the logits before the softmax. `T<1` makes the distribution sharper (deterministic), `T>1` makes it flatter (creative).
- **Top-K:** Discards all but the `k` highest probability tokens.
- **Top-P (Nucleus):** Sorts tokens by probability and discards those that fall outside the cumulative probability mass `p`.


## 3. Prompt Engineering & Optimization


### Q: Chain-of-Thought (CoT) prompting vs Tree-of-Thought (ToT).
**Answer:** CoT asks the model to "think step-by-step", forcing it to unfold its reasoning linearly. ToT allows the model to explore multiple diverging reasoning paths, evaluate them, and back-track if a path leads to a dead end, simulating a Monte Carlo Tree Search.


### Q: What is the "Lost in the Middle" phenomenon?
**Answer:** Research shows LLMs correctly recall information located at the very beginning or the very end of a long context window but severely degrade in retrieving facts buried in the middle. Mitigation involves forcing critical context to the end or using RAG to shorten the prompt.


### Q: Scenario: Few-Shot examples cause the LLM to overfit on format.
**Answer:** If you provide 3 examples that all end with a short sentence, the LLM will incorrectly assume *all* answers must be short sentences. To fix this, increase the variance in your few-shot examples (show one short, one long, one complex) to teach the model principles, not just rigid format patterns.


## 4. Retrieval-Augmented Generation (RAG)


### Q: Text chunking strategies.
**Answer:** 
- **Fixed-size:** Chops text every N tokens with some overlap. Fast but context-blind.
- **Recursive:** Tries to split on paragraphs, then sentences, then words, respecting semantic boundaries.
- **Semantic:** Uses an embedding model to find logical shifts in topics and splits there.
- **Parent-Child:** Retrieves small chunks (children) for dense matching precision, but passes the larger surrounding document (parent) to the LLM for broader context.


### Q: Hybrid Search and Reciprocal Rank Fusion (RRF).
**Answer:** Dense vector search struggles with exact keyword matching (like serial numbers). Hybrid search combines BM25 (keyword search) with Dense Vectors. RRF is the algorithm used to blend the ranked lists from both systems into a single, unified top-K list.


### Q: Scenario: RAG system returns contradictory answers.
**Answer:** When the database contains an old PDF policy and a new PDF policy, the LLM gets confused. Fix this by implementing **Metadata Filtering** (e.g., retrieving only where `year >= 2024`) or passing the dates into the prompt and explicitly instructing the LLM to privilege newer documents.


## 5. AI Agents & Agentic Systems


### Q: Scenario: Agent enters an infinite loop.
**Answer:** Agents loop when tool calls fail silently or yield identical outputs. Mitigation: Implement a hard max-iteration cap (circuit breaker). Also, pass the traceback error directly to the LLM so it knows *why* the tool failed, prompting it to alter its arguments.


## 6. Vector Databases & Embeddings
### Q: Scenario: Swapping embedding models with zero downtime.
**Answer:** You cannot compare vectors from two different models. You must:
1. Spin up a new cluster indexing the new vectors.
2. Dual-write new incoming data to both clusters.
3. Run a backfill script to re-embed the historical database into the new DB.
4. Cut over traffic once the backfill completes.


## 7. Fine-Tuning & Model Alignment
### Q: DPO vs RLHF.
**Answer:** RLHF requires training a separate Reward Model and running complex PPO algorithms. Direct Preference Optimization (DPO) simplifies this by mathematically framing the preference optimization directly as a loss function over the LLM itself, completely eliminating the need for an external reward model.


## 8. AI System Design
### Q: Design a live AI Coding Assistant.
**Answer:** 
- Latency is key: Use a fast sub-10B model.
- Streaming: Ghost text must stream instantly.
- Context: Build a local graph of the IDE. Fetch the current file, currently imported files, and actively highlighted blocks. Use Fill-In-The-Middle (FIM) training objectives so the model knows what follows the cursor.


## 9. Production AI & LLMOps
### Q: Scenario: Peak traffic hits your LLM endpoint.
**Answer:** Use an inference engine like `vLLM` equipped with continuous batching. Instead of waiting for a batch of 8 requests to all finish generation, continuous batching ejects requests as soon as they output their EOS token and instantly slots in a new request, keeping GPU utilization near 100%.
